In [2]:
# Importing libraries and loading the prepared data
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv")["disease_encoded"]
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["disease_encoded"]

label_encoder = joblib.load(
    DATA_DIR / "label_encoder.joblib"
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (80000, 174)
Testing data: (20000, 174)
Training labels: (80000,)
Testing labels: (20000,)


In [4]:
# Training Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Training completed successfully!")

Training Random Forest...
Training completed successfully!


In [6]:
# Evaluating the model
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)
weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

Accuracy:    0.3523
Macro F1:    0.2229
Weighted F1: 0.3567


In [7]:
# Generating the classification report
test_class_ids = np.unique(y_test)

print(
    classification_report(
        y_test,
        y_pred,
        labels=test_class_ids,
        target_names=label_encoder.inverse_transform(
            test_class_ids
        ),
        zero_division=0
    )
)

                               precision    recall  f1-score   support

        Acute_Gastroenteritis       0.00      0.00      0.00         1
Amyotrophic_Lateral_Sclerosis       0.16      0.50      0.24        10
                       Asthma       0.64      0.31      0.42      4259
                         COPD       0.24      0.39      0.30       412
                      COVID19       0.26      0.34      0.30       407
       Chronic_Kidney_Disease       0.39      0.37      0.38       999
        Chronic_Liver_Disease       0.00      0.00      0.00         1
                    Cirrhosis       0.00      0.00      0.00         4
                  Common_Cold       0.23      0.46      0.31       447
      Coronary_Artery_Disease       0.19      0.46      0.26       184
                Crohn_Disease       0.00      0.00      0.00         1
              Cystic_Fibrosis       0.91      0.91      0.91        33
                       Dengue       0.00      0.00      0.00         3
     

In [8]:
# Saving the trained model
model_path = MODEL_DIR / "random_forest_model.joblib"
joblib.dump(rf_model, model_path)
print(f"Model saved successfully: {model_path}")

Model saved successfully: ..\models\random_forest_model.joblib
